# DDRI 가중치 방식 샘플링 2안 노트북

이 노트북은 행을 삭제하지 않고, 반복 패턴의 영향만 줄이는 `가중치 방식`을 실행하기 위한 작업 노트다.

샘플링 2안 정의:
- 정본은 유지한다.
- 행은 삭제하지 않는다.
- `2023 train`에만 가중치를 부여한다.
- 같은 `station_id`, 같은 `weekday` 안에서 비슷한 일 패턴이 많이 반복될수록 각 날짜 가중치를 낮춘다.
- 검증 `2024`, 테스트 `2025`는 원래 분할을 그대로 쓴다.
- 임포트는 항상 별도 셀로 분리한다.


## 기준선 요약

현재 기준선 상황:
- 무샘플링 기준선은 확보했다.
- 샘플링 1안(대표일 추출)은 행 수를 줄였지만 성능 이득이 없었다.
- 따라서 다음 비교안은 행 삭제 없이 영향만 줄이는 `가중치 방식`이다.


## 1. 임포트

이 노트북도 실행 노트북이므로 임포트만 담당하는 셀을 별도로 둔다.


In [ ]:
from pathlib import Path
import importlib.util
import json
import math
import sys

import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler


## 2. 실행 설정

이 셀에서는 데이터셋, 타깃, 군집 수 비율, 가중치 공식을 정한다.


In [ ]:
ROOT = Path('/Users/cheng80/Desktop/ddri_work')

dataset_name = 'rep15'  # 'rep15' 또는 'full161'
target_col = 'bike_change_deseasonalized'  # 또는 'bike_change_raw'
cluster_ratio = 0.7  # 그룹별 대표 패턴 수를 얼마나 촘촘하게 잡을지 결정
min_days_per_group = 8  # 이보다 적은 그룹은 가중치 차등을 주지 않음
weight_rule = 'inverse_sqrt_cluster_size'  # 현재 기본 규칙

canonical_dir_map = {
    'rep15': ROOT / '3조 공유폴더' / '대표대여소_예측데이터_15개' / 'canonical_data',
    'full161': ROOT / '3조 공유폴더' / '군집별 데이터_전체 스테이션' / 'canonical_data',
}

weighted_output_dir_map = {
    'rep15': ROOT / '3조 공유폴더' / '대표대여소_예측데이터_15개' / 'sampling_plan2_outputs',
    'full161': ROOT / '3조 공유폴더' / '군집별 데이터_전체 스테이션' / 'sampling_plan2_outputs',
}

canonical_train_path = canonical_dir_map[dataset_name] / 'ddri_prediction_canonical_train_2023_2024.csv'
canonical_test_path = canonical_dir_map[dataset_name] / 'ddri_prediction_canonical_test_2025.csv'
weighted_output_dir = weighted_output_dir_map[dataset_name]
weighted_output_dir


## 3. 정본에서 2023 train만 로드


In [ ]:
train_df = pd.read_csv(canonical_train_path)
train_df['date'] = pd.to_datetime(train_df['date'])
train_2023 = train_df[train_df['date'].dt.year == 2023].copy()

print('rows:', len(train_2023))
print('stations:', train_2023['station_id'].nunique())
print('date range:', train_2023['date'].min(), '~', train_2023['date'].max())


## 4. 일단위 24시간 프로파일 생성


In [ ]:
daily_profile = (
    train_2023
    .pivot_table(
        index=['station_id', 'date', 'weekday'],
        columns='hour',
        values=target_col,
        aggfunc='mean',
    )
    .reset_index()
)

daily_profile.columns = [f'hour_{c}' if isinstance(c, (int, np.integer)) else c for c in daily_profile.columns]
profile_cols = [c for c in daily_profile.columns if c.startswith('hour_')]

print('daily profiles:', len(daily_profile))
daily_profile.head()


## 5. 가중치 부여 함수

핵심 원칙:
- 그룹이 너무 작으면 모든 날짜 가중치를 1로 둔다.
- 같은 `station_id`, 같은 `weekday` 안에서 유사한 날짜를 군집화한다.
- 같은 군집에 날짜가 많이 몰릴수록 각 날짜 가중치를 낮춘다.
- 현재 기본 공식은 `1 / sqrt(cluster_size)`이며, 마지막에 전체 평균이 1이 되도록 정규화한다.


In [ ]:
def build_day_weights(group: pd.DataFrame, cluster_ratio: float, min_days_per_group: int) -> pd.DataFrame:
    group = group.copy().reset_index(drop=True)

    if len(group) < min_days_per_group:
        group['cluster_id'] = np.arange(len(group))
        group['cluster_size'] = 1
        group['day_weight_raw'] = 1.0
        return group

    keep_count = max(1, math.ceil(len(group) * cluster_ratio))
    keep_count = min(keep_count, len(group))

    feature = group[profile_cols].fillna(0.0).to_numpy(dtype=float)
    scaled = StandardScaler().fit_transform(feature)
    model = KMeans(n_clusters=keep_count, random_state=42, n_init=10)
    labels = model.fit_predict(scaled)

    group['cluster_id'] = labels
    cluster_size = group.groupby('cluster_id')['date'].transform('size')
    group['cluster_size'] = cluster_size.astype(int)
    group['day_weight_raw'] = 1.0 / np.sqrt(group['cluster_size'].astype(float))
    return group


## 6. 날짜 가중치 계산


In [ ]:
weighted_daily = pd.concat(
    [
        build_day_weights(g, cluster_ratio=cluster_ratio, min_days_per_group=min_days_per_group)
        for _, g in daily_profile.groupby(['station_id', 'weekday'], sort=False)
    ],
    ignore_index=True,
)

weighted_daily['day_weight'] = weighted_daily['day_weight_raw'] / weighted_daily['day_weight_raw'].mean()

weight_summary = pd.DataFrame([
    {
        'dataset': dataset_name,
        'target': target_col,
        'train_rows_before': int(len(train_2023)),
        'train_rows_after': int(len(train_2023)),
        'days_before': int(len(daily_profile)),
        'days_after': int(len(weighted_daily)),
        'weight_min': float(weighted_daily['day_weight'].min()),
        'weight_mean': float(weighted_daily['day_weight'].mean()),
        'weight_max': float(weighted_daily['day_weight'].max()),
    }
])

weight_summary


## 7. 날짜 가중치를 시간행으로 확장

학습 모델은 시간행 단위로 학습하므로, 날짜 가중치를 원래 시간행에 붙인다.


In [ ]:
train_weighted = train_2023.merge(
    weighted_daily[['station_id', 'date', 'day_weight', 'cluster_id', 'cluster_size']],
    on=['station_id', 'date'],
    how='left',
)

train_weighted['day_weight'] = train_weighted['day_weight'].fillna(1.0)
train_weighted[['station_id', 'date', 'hour', 'day_weight', 'cluster_id', 'cluster_size']].head(20)


## 8. 가중치 분포 확인 셀


In [ ]:
pd.DataFrame([
    {
        'rows': int(len(train_weighted)),
        'weight_min': float(train_weighted['day_weight'].min()),
        'weight_p10': float(train_weighted['day_weight'].quantile(0.10)),
        'weight_p50': float(train_weighted['day_weight'].quantile(0.50)),
        'weight_p90': float(train_weighted['day_weight'].quantile(0.90)),
        'weight_max': float(train_weighted['day_weight'].max()),
    }
])


## 9. 학습용 모듈 로드


In [ ]:
WORK_DIR = ROOT / 'works' / '06_prediction_training'
FEATURE_SCRIPT = WORK_DIR / '05_scripts' / '03_ddri_prediction_modeling_feature_builder.py'
MODEL_SCRIPT = WORK_DIR / '05_scripts' / '04_ddri_prediction_train_eval.py'

feature_spec = importlib.util.spec_from_file_location('feature_builder_sampling_plan2', FEATURE_SCRIPT)
feature_builder = importlib.util.module_from_spec(feature_spec)
sys.modules[feature_spec.name] = feature_builder
feature_spec.loader.exec_module(feature_builder)

train_spec = importlib.util.spec_from_file_location('train_eval_sampling_plan2', MODEL_SCRIPT)
train_eval = importlib.util.module_from_spec(train_spec)
sys.modules[train_spec.name] = train_eval
train_spec.loader.exec_module(train_eval)

pd.DataFrame({'candidate_model': [name for name, _ in train_eval.candidate_models()]})


## 10. 검증셋과 테스트셋 준비


In [ ]:
valid_2024 = train_df[train_df['date'].dt.year == 2024].copy()
test_df = pd.read_csv(canonical_test_path)
test_df['date'] = pd.to_datetime(test_df['date'])
test_2025 = test_df.copy()

train_model = feature_builder.add_missing_flags_and_fill(train_weighted)
valid_model = feature_builder.add_missing_flags_and_fill(valid_2024)
test_model = feature_builder.add_missing_flags_and_fill(test_2025)

pd.DataFrame([
    {'frame': 'train_model_weighted', 'rows': len(train_model), 'target_null_rows': int(train_model[target_col].isna().sum())},
    {'frame': 'valid_model', 'rows': len(valid_model), 'target_null_rows': int(valid_model[target_col].isna().sum())},
    {'frame': 'test_model', 'rows': len(test_model), 'target_null_rows': int(test_model[target_col].isna().sum())},
])


## 11. 가중치 전용 modeling_data 저장


In [ ]:
weighted_modeling_dir = weighted_output_dir / 'modeling_data'
weighted_training_dir = weighted_output_dir / 'training_runs'
weighted_modeling_dir.mkdir(parents=True, exist_ok=True)
weighted_training_dir.mkdir(parents=True, exist_ok=True)

train_model_save = train_model.copy()
valid_model_save = valid_model.copy()
test_model_save = test_model.copy()
for frame in [train_model_save, valid_model_save, test_model_save]:
    frame['date'] = pd.to_datetime(frame['date']).dt.strftime('%Y-%m-%d')

weighted_train_model_path = weighted_modeling_dir / 'ddri_prediction_modeling_train_2023_weighted_plan2.csv'
weighted_valid_model_path = weighted_modeling_dir / 'ddri_prediction_modeling_valid_2024.csv'
weighted_test_model_path = weighted_modeling_dir / 'ddri_prediction_modeling_test_2025.csv'
weighted_meta_path = weighted_modeling_dir / 'ddri_prediction_modeling_meta_plan2.json'

train_model_save.to_csv(weighted_train_model_path, index=False, encoding='utf-8-sig')
valid_model_save.to_csv(weighted_valid_model_path, index=False, encoding='utf-8-sig')
test_model_save.to_csv(weighted_test_model_path, index=False, encoding='utf-8-sig')

weighted_meta = {
    'dataset': dataset_name,
    'sampling_plan': 'plan2_weighting',
    'target': target_col,
    'cluster_ratio': cluster_ratio,
    'min_days_per_group': min_days_per_group,
    'weight_rule': weight_rule,
    'train_rows': int(len(train_model)),
    'valid_rows': int(len(valid_model)),
    'test_rows': int(len(test_model)),
}
weighted_meta_path.write_text(json.dumps(weighted_meta, ensure_ascii=False, indent=2), encoding='utf-8')

pd.DataFrame([
    {'name': 'weighted_train_model', 'path': str(weighted_train_model_path)},
    {'name': 'valid_model', 'path': str(weighted_valid_model_path)},
    {'name': 'test_model', 'path': str(weighted_test_model_path)},
])


## 12. 가중치 학습 입력 확정


In [ ]:
train_fit = train_model[train_model[target_col].notna()].copy()
valid_fit = valid_model[valid_model[target_col].notna()].copy()
test_fit = test_model[test_model[target_col].notna()].copy()

feature_cols = train_eval.build_feature_cols(train_fit)
feature_cols = [c for c in feature_cols if c not in {'day_weight', 'cluster_id', 'cluster_size'}]

X_train = train_fit[feature_cols]
y_train = train_fit[target_col]
train_weights = train_fit['day_weight'].astype(float)
X_valid = valid_fit[feature_cols]
y_valid = valid_fit[target_col]
X_test = test_fit[feature_cols]
y_test = test_fit[target_col]

pd.DataFrame([
    {'frame': 'train_fit_weighted', 'rows_used': len(train_fit), 'feature_count': len(feature_cols), 'weight_mean': float(train_weights.mean())},
    {'frame': 'valid_fit', 'rows_used': len(valid_fit), 'feature_count': len(feature_cols), 'weight_mean': None},
    {'frame': 'test_fit', 'rows_used': len(test_fit), 'feature_count': len(feature_cols), 'weight_mean': None},
])


## 13. 2024 validation 모델 비교

여기서는 후보 모델 모두에 `sample_weight`를 넣어 다시 비교한다.


In [ ]:
selection_rows = []
for name, model in train_eval.candidate_models():
    model.fit(X_train, y_train, sample_weight=train_weights)
    pred = model.predict(X_valid)
    selection_rows.append({'model': name, **train_eval.metrics(y_valid, pred)})

selection_df = pd.DataFrame(selection_rows).sort_values(['rmse', 'mae', 'r2'], ascending=[True, True, False]).reset_index(drop=True)
selection_df


## 14. 우세 모델 선택


In [ ]:
best_name = selection_df.iloc[0]['model']
best_name


## 15. 2023+2024 재학습 후 2025 테스트

최종 재학습에서도 train 구간에만 가중치를 주고, `2024 valid`는 가중치 1로 결합한다.


In [ ]:
final_X = pd.concat([X_train, X_valid], ignore_index=True)
final_y = pd.concat([y_train, y_valid], ignore_index=True)
final_weights = pd.concat([
    train_weights.reset_index(drop=True),
    pd.Series(np.ones(len(X_valid)), dtype=float),
], ignore_index=True)

final_model = dict(train_eval.candidate_models())[best_name]
final_model.fit(final_X, final_y, sample_weight=final_weights)
test_pred = final_model.predict(X_test)
test_metrics = train_eval.metrics(y_test, test_pred)
pd.DataFrame([{'model': best_name, **test_metrics}])


## 16. 가중치 2안 결과 저장


In [ ]:
selection_path = weighted_training_dir / f'ddri_{dataset_name}_{target_col}_selection_metrics_plan2.csv'
test_metrics_path = weighted_training_dir / f'ddri_{dataset_name}_{target_col}_test_metrics_plan2.csv'
pred_path = weighted_training_dir / f'ddri_{dataset_name}_{target_col}_test_predictions_plan2.csv'
meta_path = weighted_training_dir / f'ddri_{dataset_name}_{target_col}_training_meta_plan2.json'

selection_df.to_csv(selection_path, index=False, encoding='utf-8-sig')
pd.DataFrame([{'model': best_name, **test_metrics}]).to_csv(test_metrics_path, index=False, encoding='utf-8-sig')
pred_df = test_fit[['station_id', 'date', 'hour', target_col]].copy()
pred_df['prediction'] = test_pred
pred_df.to_csv(pred_path, index=False, encoding='utf-8-sig')

run_meta = {
    'dataset': dataset_name,
    'sampling_plan': 'plan2_weighting',
    'target': target_col,
    'cluster_ratio': cluster_ratio,
    'min_days_per_group': min_days_per_group,
    'weight_rule': weight_rule,
    'train_rows_used': int(len(train_fit)),
    'valid_rows_used': int(len(valid_fit)),
    'test_rows_used': int(len(test_fit)),
    'best_model': best_name,
    'selection_output': str(selection_path),
    'test_metrics_output': str(test_metrics_path),
    'test_predictions_output': str(pred_path),
}
meta_path.write_text(json.dumps(run_meta, ensure_ascii=False, indent=2), encoding='utf-8')

pd.DataFrame([
    {'name': 'selection_metrics', 'path': str(selection_path)},
    {'name': 'test_metrics', 'path': str(test_metrics_path)},
    {'name': 'test_predictions', 'path': str(pred_path)},
    {'name': 'run_meta', 'path': str(meta_path)},
])
